# Child Mortality Analysis
Predicting Under-5 Mortality Across Countries Using World Development Indicators
Group 5 of "Einführung in maschinelles Lernen" - A Data Science Project.

Ayoub Taychi

Hajar Lasri

Yama Saputra

# 1.1 Domain Problem
# Research Questions
## 1. Can ML regression models predict under-5 child mortality rates across countries using health, education, income, sanitation, and governance indicators from the WDI?
Justification: Under-5 mortality is a socially relevant and policy-important development outcome that is strontly linked to health, sanitation, income, and demographic conditions.

## 2. Can ML classifiers predict whether a country-year will fall into a high under-5 mortality risk group in the following year?
Justification: High risk is defined as the top 25% next-year under-5 mortality values, which creates a simple and interpretable binary target.

Technical Environment: Python v3.13.13 Importing data project libraries, pandas and scikit learn.

In [2]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import sklearn
import pycountry

print(f"pandas: {pd.__version__}")
print(f"numpy: {np.__version__}")
print(f"matplotlib: {matplotlib.__version__}")
print(f"sklearn: {sklearn.__version__}")
print(f"pycountry: {pycountry.__version__}")

pandas: 2.2.3
numpy: 1.26.4
matplotlib: 3.9.2
sklearn: 1.5.1
pycountry: 26.2.16


# Child Mortality Analysis
## Stage 1 — Data Preparation

This section prepares the WDI data for later analysis by:
- renaming key metadata fields,
- retaining countries only,
- selecting candidate indicators,
- reshaping the data into a country-year panel.

# Stage 1 — Data Preparation

In [5]:
df = pd.read_csv("WB_WDI_WIDEF (1).csv")
print(df.shape)
df.head()
df.info()
print(df.columns.tolist())

(295181, 107)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 295181 entries, 0 to 295180
Columns: 107 entries, STRUCTURE to 2025
dtypes: float64(66), int64(2), object(39)
memory usage: 241.0+ MB
['STRUCTURE', 'STRUCTURE_ID', 'ACTION', 'FREQ', 'REF_AREA', 'INDICATOR', 'SEX', 'AGE', 'URBANISATION', 'UNIT_MEASURE', 'COMP_BREAKDOWN_1', 'COMP_BREAKDOWN_2', 'COMP_BREAKDOWN_3', 'AGG_METHOD', 'UNIT_TYPE', 'DECIMALS', 'DATABASE_ID', 'TIME_FORMAT', 'COMMENT_TS', 'UNIT_MULT', 'DATA_SOURCE', 'OBS_CONF', 'OBS_STATUS', 'FREQ_LABEL', 'REF_AREA_LABEL', 'INDICATOR_LABEL', 'SEX_LABEL', 'AGE_LABEL', 'URBANISATION_LABEL', 'UNIT_MEASURE_LABEL', 'COMP_BREAKDOWN_1_LABEL', 'COMP_BREAKDOWN_2_LABEL', 'COMP_BREAKDOWN_3_LABEL', 'AGG_METHOD_LABEL', 'UNIT_TYPE_LABEL', 'DECIMALS_LABEL', 'DATABASE_ID_LABEL', 'TIME_FORMAT_LABEL', 'UNIT_MULT_LABEL', 'OBS_STATUS_LABEL', 'OBS_CONF_LABEL', '1960', '1961', '1962', '1963', '1964', '1965', '1966', '1967', '1968', '1969', '1970', '1971', '1972', '1973', '1974', '1975', '19

In [ ]:
df_prep = df.rename(columns={
    "REF_AREA": "country_code",
    "REF_AREA_LABEL": "country_name",
    "INDICATOR": "indicator_code",
    "INDICATOR_LABEL": "indicator_name"
}).copy()

df_prep[["country_code", "country_name", "indicator_code", "indicator_name"]].head()

In [ ]:
year_cols = [col for col in df_prep.columns if col.isdigit()]

print("Number of year columns:", len(year_cols))
print("First 5 years:", year_cols[:5])
print("Last 5 years:", year_cols[-5:])

In [ ]:
countries_table = (
    df_prep[["country_code", "country_name"]]
    .drop_duplicates()
    .sort_values(["country_name", "country_code"])
    .reset_index(drop=True)
)

print("Number of unique entities:", countries_table.shape[0])
countries_table.head(20)

In [ ]:
iso3 = {c.alpha_3 for c in pycountry.countries} | {"XKX"}

countries_only_table = (
    countries_table[countries_table["country_code"].isin(iso3)]
    .copy()
    .sort_values(["country_name", "country_code"])
    .reset_index(drop=True)
)

print("Number of countries only:", countries_only_table.shape[0])
countries_only_table.head(20)

In [ ]:
indicators_table = (
    df_prep[["indicator_code", "indicator_name"]]
    .drop_duplicates()
    .sort_values(["indicator_name", "indicator_code"])
    .reset_index(drop=True)
)

print("Number of unique indicators:", indicators_table.shape[0])
indicators_table.head(20)

In [ ]:
keywords = [
    "under-5",
    "mortality",
    "gdp per capita",
    "gini",
    "health expenditure",
    "physicians",
    "immunization",
    "water",
    "sanitation",
    "electricity",
    "primary completion",
    "literacy",
    "rule of law",
    "government effectiveness",
    "fertility",
    "urban population",
    "population ages 0-14",
    "life expectancy"
]

for kw in keywords:
    print(f"\n========== {kw.upper()} ==========")
    display(
        indicators_table[
            indicators_table["indicator_name"].str.contains(kw, case=False, na=False)
        ].head(20)
    )

## Candidate Indicator Selection

We do not use the full WDI indicator universe.  
Instead, we define a broader candidate set based on theoretical relevance to child mortality, including health, income, sanitation, infrastructure, and demographic variables.

In [ ]:
candidate_indicator_dict = {
    "u5mr": "WB_WDI_SH_DYN_MORT",

    "gdp_pc_constant_2015_usd": "WB_WDI_NY_GDP_PCAP_KD",
    "gdp_pc_ppp_current": "WB_WDI_NY_GDP_PCAP_PP_CD",
    "gini": "WB_WDI_SI_POV_GINI",

    "health_exp_pct_gdp": "WB_WDI_SH_XPD_CHEX_GD_ZS",
    "physicians_per_1000": "WB_WDI_SH_MED_PHYS_ZS",
    "dpt": "WB_WDI_SH_IMM_IDPT",
    "measles": "WB_WDI_SH_IMM_MEAS",
    "life_expectancy": "WB_WDI_SP_DYN_LE00_IN",
    "neonatal_mortality": "WB_WDI_SH_DYN_NMRT",

    "water_basic": "WB_WDI_SH_H2O_BASW_ZS",
    "water_safe": "WB_WDI_SH_H2O_SMDW_ZS",
    "sanitation_basic": "WB_WDI_SH_STA_BASS_ZS",
    "sanitation_safe": "WB_WDI_SH_STA_SMSS_ZS",
    "electricity": "WB_WDI_EG_ELC_ACCS_ZS",

    "primary_completion": "WB_WDI_SE_PRM_CMPT_ZS",
    "adult_literacy": "WB_WDI_SE_ADT_LITR_ZS",
    "rule_of_law": "WB_WDI_RL_EST",
    "govt_effectiveness": "WB_WDI_GE_EST",

    "fertility": "WB_WDI_SP_DYN_TFRT_IN",
    "urban_pct": "WB_WDI_SP_URB_TOTL_IN_ZS",
    "urban_population_total": "WB_WDI_SP_URB_TOTL",
    "population_total": "WB_WDI_SP_POP_TOTL",
    "pop_0_14_pct": "WB_WDI_SP_POP_0014_TO_ZS",
}

print("Number of candidate indicators:", len(candidate_indicator_dict))
print(list(candidate_indicator_dict.keys()))

In [ ]:
candidate_indicators_table = pd.DataFrame({
    "new_name": list(candidate_indicator_dict.keys()),
    "indicator_code": list(candidate_indicator_dict.values())
})

candidate_indicators_table = candidate_indicators_table.merge(
    indicators_table,
    on="indicator_code",
    how="left"
).sort_values("new_name").reset_index(drop=True)

candidate_indicators_table

In [ ]:
keep_cols = ["country_code", "country_name", "indicator_code", "indicator_name"] + year_cols
df_stage1 = df_prep[keep_cols].copy()

print("Initial stage1 shape:", df_stage1.shape)
df_stage1.head(10)

In [ ]:
country_codes_only = set(countries_only_table["country_code"])

df_stage1 = df_stage1[df_stage1["country_code"].isin(country_codes_only)].copy()

print("Shape after countries-only filter:", df_stage1.shape)
print("Unique countries:", df_stage1["country_code"].nunique())

In [ ]:
df_stage1 = df_stage1[df_stage1["indicator_code"].isin(candidate_indicator_dict.values())].copy()

print("Shape after candidate-indicator filter:", df_stage1.shape)
print("Unique indicators kept:", df_stage1["indicator_code"].nunique())
sorted(df_stage1["indicator_code"].unique())

In [ ]:
inverse_indicator_map = {v: k for k, v in candidate_indicator_dict.items()}

print("Inverse map size:", len(inverse_indicator_map))
list(inverse_indicator_map.items())[:5]

In [ ]:
long_df = df_stage1.melt(
    id_vars=["country_code", "country_name", "indicator_code", "indicator_name"],
    value_vars=year_cols,
    var_name="year",
    value_name="value"
)

long_df["year"] = long_df["year"].astype(int)
long_df["value"] = pd.to_numeric(long_df["value"], errors="coerce")

print("long_df shape:", long_df.shape)
display(long_df.head(10))

In [ ]:
long_df["indicator_short"] = long_df["indicator_code"].map(inverse_indicator_map)

print("Missing short names:", long_df["indicator_short"].isna().sum())
display(
    long_df[["indicator_code", "indicator_short"]]
    .drop_duplicates()
    .sort_values("indicator_short")
    .head(30)
)

In [ ]:
dup_long = long_df.duplicated(
    subset=["country_code", "country_name", "year", "indicator_short"]
).sum()

print("Duplicate country-year-indicator rows before pivot:", dup_long)

if dup_long > 0:
    display(
        long_df[
            long_df.duplicated(
                subset=["country_code", "country_name", "year", "indicator_short"],
                keep=False
            )
        ].sort_values(["country_name", "year", "indicator_short"]).head(20)
    )

In [ ]:
assert dup_long == 0, "Resolve duplicates before pivoting."

panel_candidate = long_df.pivot(
    index=["country_code", "country_name", "year"],
    columns="indicator_short",
    values="value"
).reset_index()

panel_candidate.columns.name = None

print("panel_candidate shape:", panel_candidate.shape)
display(panel_candidate.head(10))

In [ ]:
df_stage1.to_csv("stage1_filtered_raw.csv", index=False)
long_df.to_csv("stage1_long.csv", index=False)
panel_candidate.to_csv("stage1_panel_candidate_full_horizon.csv", index=False)

print("Stage 1 files saved successfully.")

#  End of Stage 1
The raw WDI data were prepared into a country-year candidate panel by:
- renaming key metadata columns,
- keeping countries only,
- selecting a broader candidate indicator set,
- reshaping the data from wide format to long format,
- and building a country-year panel for the full available time horizon.

No final time restriction has been applied yet. The choice of the final modeling window will be justified in Stage 2 using data quality, missingness, and coverage analysis.

# Stage 2 = Data Cleaning + Data Quality Audit + Early EDA 

In [ ]:
dup_count = panel_candidate.duplicated(subset=["country_code", "year"]).sum()

print("Duplicate country-year rows:", dup_count)

In [ ]:
missing_report = pd.DataFrame({
    "missing_count": panel_candidate.isna().sum(),
    "missing_pct": (panel_candidate.isna().mean() * 100)
}).sort_values("missing_pct", ascending=False)

missing_report

In [ ]:
target_coverage_by_year = (
    panel_candidate.groupby("year")["u5mr"]
    .apply(lambda s: s.notna().sum())
    .reset_index(name="u5mr_non_missing_countries")
)

target_coverage_by_year.head(66)

In [ ]:
predictor_cols = [
    "gdp_pc_constant_2015_usd",
    "gdp_pc_ppp_current",
    "gini",
    "health_exp_pct_gdp",
    "physicians_per_1000",
    "dpt",
    "measles",
    "life_expectancy",
    "neonatal_mortality",
    "water_basic",
    "water_safe",
    "sanitation_basic",
    "sanitation_safe",
    "electricity",
    "primary_completion",
    "adult_literacy",
    "rule_of_law",
    "govt_effectiveness",
    "fertility",
    "urban_pct",
    "urban_population_total",
    "population_total",
    "pop_0_14_pct"
]

predictor_coverage_by_year = panel_candidate.groupby("year")[predictor_cols].apply(
    lambda df: df.notna().sum()
)

predictor_coverage_by_year.head(20)

In [ ]:
core_candidate_cols = [
    "u5mr",
    "gdp_pc_constant_2015_usd",
    "health_exp_pct_gdp",
    "dpt",
    "measles",
    "water_basic",
    "sanitation_basic",
    "electricity",
    "fertility",
    "urban_pct",
    "pop_0_14_pct"
]

joint_core_by_year = (
    panel_candidate.groupby("year")
    .apply(lambda df: pd.Series({
        "n_countries_total": df["country_code"].nunique(),
        "n_complete_core_rows": df[core_candidate_cols].notna().all(axis=1).sum()
    }))
    .reset_index()
)

joint_core_by_year["complete_core_pct"] = (
    100 * joint_core_by_year["n_complete_core_rows"] / joint_core_by_year["n_countries_total"]
)

joint_core_by_year.head(20)

In [ ]:
joint_core_by_year.tail(20)

In [ ]:
print("First year with > 0 complete core rows:",
      joint_core_by_year.loc[joint_core_by_year["n_complete_core_rows"] > 0, "year"].min())

print("First year with >= 50% complete core rows:",
      joint_core_by_year.loc[joint_core_by_year["complete_core_pct"] >= 50, "year"].min())

print("First year with >= 70% complete core rows:",
      joint_core_by_year.loc[joint_core_by_year["complete_core_pct"] >= 70, "year"].min())

print("First year with >= 80% complete core rows:",
      joint_core_by_year.loc[joint_core_by_year["complete_core_pct"] >= 80, "year"].min())

In [ ]:
joint_core_by_year[
    (joint_core_by_year["year"] >= 1990) &
    (joint_core_by_year["year"] <= 2025)
]

## Time-Window Decision

We explored the full 1960–2025 horizon, but joint completeness for the target and the core predictors is effectively zero before 2000.

From 2000 onward, the candidate panel becomes usable and remains stable through 2023.

Therefore, the final modeling window will be restricted to 2000–2023.

In [ ]:

plt.figure(figsize=(12, 6))
plt.plot(
    joint_core_by_year["year"],
    joint_core_by_year["complete_core_pct"],
    marker="o"
)

plt.axvline(2000, linestyle="--", color="red", label="Window start: 2000")
plt.axvline(2023, linestyle="--", color="green", label="Window end: 2023")
plt.legend(fontsize=10)

plt.title("Joint completeness of target + core predictors by year")
plt.xlabel("Year")
plt.ylabel("Complete core rows (%)")
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(
    target_coverage_by_year["year"],
    target_coverage_by_year["u5mr_non_missing_countries"],
    marker="o"
)

plt.axvline(2000, linestyle="--", color="red", label="Window start: 2000")
plt.axvline(2023, linestyle="--", color="green", label="Window end: 2023")
plt.legend(fontsize=10)

plt.title("Under-5 mortality target availability by year")
plt.xlabel("Year")
plt.ylabel("Countries with observed under-5 mortality")
plt.grid(True)
plt.show()

In [ ]:
panel_modern = panel_candidate[
    (panel_candidate["year"] >= 2000) &
    (panel_candidate["year"] <= 2023)
].copy()

print("Modern panel shape:", panel_modern.shape)
print("Year range:", panel_modern["year"].min(), "to", panel_modern["year"].max())
print("Countries:", panel_modern["country_code"].nunique())

In [ ]:
modern_missing_report = pd.DataFrame({
    "missing_count": panel_modern.isna().sum(),
    "missing_pct": panel_modern.isna().mean() * 100
}).sort_values("missing_pct", ascending=False)

modern_missing_report

## Country-Level Completeness Audit

This step summarizes how many usable observations each country contributes in the 2000–2023 modern panel.

It is used for descriptive assessment of panel quality, not for aggressive country-level exclusion from the main model.

In [ ]:
country_quality = (
    panel_modern.groupby(["country_code", "country_name"], as_index=False)
    .agg(
        total_years=("year", "nunique"),
        u5mr_observed_years=("u5mr", lambda s: s.notna().sum()),
        complete_core_years=("u5mr", lambda s: 0)
    )
)

complete_core_counts = (
    panel_modern.assign(
        complete_core=panel_modern[["u5mr"] + core_candidate_cols].notna().all(axis=1)
    )
    .groupby(["country_code", "country_name"], as_index=False)["complete_core"]
    .sum()
    .rename(columns={"complete_core": "complete_core_years"})
)

country_quality = country_quality.drop(columns=["complete_core_years"]).merge(
    complete_core_counts,
    on=["country_code", "country_name"],
    how="left"
)

country_quality["u5mr_missing_years"] = country_quality["total_years"] - country_quality["u5mr_observed_years"]
country_quality["u5mr_coverage_pct"] = 100 * country_quality["u5mr_observed_years"] / country_quality["total_years"]
country_quality["complete_core_pct"] = 100 * country_quality["complete_core_years"] / country_quality["total_years"]

print("Countries with full 24-year core coverage:",
      (country_quality["complete_core_years"] == 24).sum())
print("Countries with < 10 complete core years:",
      (country_quality["complete_core_years"] < 10).sum())

display(country_quality.sort_values("complete_core_years", ascending=False).head(20))

## Broad Feature Audit

This audit is intentionally broader than the final modeling feature set.

Some variables are still examined here for diagnostic and EDA purposes, even if they will later be excluded from the main model because of missingness, redundancy, or conceptual proximity to the target.

**Note:** life expectancy and neonatal mortality are included for audit reference only and are excluded from all modeling because of their conceptual proximity to the target.

In [ ]:
feature_cols_for_audit = [
    "u5mr",
    "gdp_pc_constant_2015_usd",
    "gdp_pc_ppp_current",
    "gini",
    "health_exp_pct_gdp",
    "physicians_per_1000",
    "dpt",
    "measles",
    "life_expectancy",
    "neonatal_mortality",
    "water_basic",
    "water_safe",
    "sanitation_basic",
    "sanitation_safe",
    "electricity",
    "primary_completion",
    "adult_literacy",
    "rule_of_law",
    "govt_effectiveness",
    "fertility",
    "urban_pct",
    "urban_population_total",
    "population_total",
    "pop_0_14_pct"
]

variation_report = pd.DataFrame({
    "non_missing_count": panel_modern[feature_cols_for_audit].notna().sum(),
    "missing_pct": panel_modern[feature_cols_for_audit].isna().mean() * 100,
    "n_unique": panel_modern[feature_cols_for_audit].nunique(dropna=True),
    "std": panel_modern[feature_cols_for_audit].std(),
    "min": panel_modern[feature_cols_for_audit].min(),
    "median": panel_modern[feature_cols_for_audit].median(),
    "max": panel_modern[feature_cols_for_audit].max()
}).sort_values(["missing_pct", "std"], ascending=[True, False])

variation_report

In [ ]:
feature_decision_table = pd.DataFrame({
    "feature": [
        "gdp_pc_constant_2015_usd",
        "health_exp_pct_gdp",
        "dpt",
        "measles",
        "water_basic",
        "sanitation_basic",
        "electricity",
        "fertility",
        "urban_pct",
        "pop_0_14_pct",

        "gdp_pc_ppp_current",
        "physicians_per_1000",
        "primary_completion",
        "rule_of_law",
        "govt_effectiveness",
        "water_safe",
        "sanitation_safe",

        "gini",
        "adult_literacy",
        "life_expectancy",
        "neonatal_mortality",
        "population_total",
        "urban_population_total"
    ],
    "proposed_role": [
        "core_main",
        "core_main",
        "core_main",
        "core_main",
        "core_main",
        "core_main",
        "core_main",
        "core_main",
        "core_main",
        "core_main",

        "extended",
        "extended",
        "extended",
        "extended",
        "extended",
        "extended",
        "extended",

        "exclude_main",
        "exclude_main",
        "exclude_main",
        "exclude_main",
        "exclude_main",
        "exclude_main"
    ],
    "reason": [
        "strong coverage and conceptually central",
        "strong coverage and conceptually central",
        "strong coverage and conceptually central",
        "strong coverage and conceptually central",
        "strong coverage and conceptually central",
        "strong coverage and conceptually central",
        "strong coverage and conceptually central",
        "strong coverage and conceptually central",
        "strong coverage and conceptually central",
        "strong coverage and conceptually central",

        "usable but more optional / redundant",
        "moderate missingness",
        "moderate missingness",
        "moderate missingness, governance extension",
        "moderate missingness, governance extension",
        "moderate missingness, optional stricter access metric",
        "moderate missingness, optional stricter access metric",

        "too much missingness",
        "too much missingness",
        "too close to target conceptually",
        "too close to target conceptually",
        "size variable, not main substantive signal",
        "size variable, not main substantive signal"
    ]
})

feature_decision_table = feature_decision_table.merge(
    variation_report[["missing_pct", "n_unique", "std"]],
    left_on="feature",
    right_index=True,
    how="left"
).sort_values(["proposed_role", "missing_pct"])

feature_decision_table

In [ ]:
core_plus_target = [
    "u5mr",
    "gdp_pc_constant_2015_usd",
    "health_exp_pct_gdp",
    "dpt",
    "measles",
    "water_basic",
    "sanitation_basic",
    "electricity",
    "fertility",
    "urban_pct",
    "pop_0_14_pct"
]

corr_core = panel_modern[core_plus_target].corr()

plt.figure(figsize=(11, 9))
plt.imshow(corr_core, aspect="auto", cmap="RdYlGn", vmin=-1, vmax=1)
plt.colorbar(label="Correlation")

plt.xticks(range(len(corr_core.columns)), corr_core.columns, rotation=90)
plt.yticks(range(len(corr_core.index)), corr_core.index)

for i in range(len(corr_core.index)):
    for j in range(len(corr_core.columns)):
        plt.text(
            j, i,
            f"{corr_core.iloc[i, j]:.2f}",
            ha="center", va="center", fontsize=8
        )

plt.title("Correlation heatmap: target + core main features")
plt.tight_layout()
plt.show()

In [ ]:
u5mr_skew_raw = panel_modern["u5mr"].dropna().skew()
u5mr_skew_log = np.log(panel_modern["u5mr"].dropna()).skew()

print("Skewness of raw u5mr:", round(u5mr_skew_raw, 3))
print("Skewness of log(u5mr):", round(u5mr_skew_log, 3))

In [ ]:
u5mr_desc = panel_modern["u5mr"].dropna().describe(
    percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]
)
u5mr_desc

In [ ]:
q1 = panel_modern["u5mr"].dropna().quantile(0.25)
q3 = panel_modern["u5mr"].dropna().quantile(0.75)
iqr = q3 - q1
upper_bound = q3 + 1.5 * iqr

print("Potential high-end outlier threshold (IQR rule):", round(upper_bound, 2))
print("Number of values above threshold:", (panel_modern["u5mr"] > upper_bound).sum())

In [ ]:
mean_val = panel_modern["u5mr"].dropna().mean()
median_val = panel_modern["u5mr"].dropna().median()

plt.figure(figsize=(10, 5))
plt.hist(panel_modern["u5mr"].dropna(), bins=30)

plt.axvline(mean_val, color="red", linestyle="--", label=f"Mean: {mean_val:.1f}")
plt.axvline(median_val, color="orange", linestyle="--", label=f"Median: {median_val:.1f}")
plt.legend()

plt.title("Distribution of under-5 mortality (raw scale)")
plt.xlabel("Under-5 mortality per 1,000 live births")
plt.ylabel("Frequency")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
panel_modern["log_u5mr"] = np.log(panel_modern["u5mr"])

mean_log = panel_modern["log_u5mr"].dropna().mean()
median_log = panel_modern["log_u5mr"].dropna().median()

plt.figure(figsize=(10, 5))
plt.hist(panel_modern["log_u5mr"].dropna(), bins=30)

plt.axvline(mean_log, color="red", linestyle="--", label=f"Mean: {mean_log:.2f}")
plt.axvline(median_log, color="orange", linestyle="--", label=f"Median: {median_log:.2f}")
plt.legend()

plt.title("Distribution of under-5 mortality (log scale)")
plt.xlabel("Log under-5 mortality")
plt.ylabel("Frequency")
plt.grid(True)
plt.tight_layout()
plt.show()

## Target Distribution Conclusion

The raw under-5 mortality distribution is strongly right-skewed, with a long upper tail and a substantial number of high-end observations.  
After log transformation, the distribution becomes substantially more symmetric and better suited for regression modeling.  
This supports the use of **log under-5 mortality** as the main continuous target in the final regression setup.

In [ ]:
core_main_features = [
    "gdp_pc_constant_2015_usd",
    "health_exp_pct_gdp",
    "dpt",
    "measles",
    "water_basic",
    "sanitation_basic",
    "electricity",
    "fertility",
    "urban_pct",
    "pop_0_14_pct"
]

extended_features = [
    "gdp_pc_ppp_current",
    "physicians_per_1000",
    "primary_completion",
    "rule_of_law",
    "govt_effectiveness",
    "water_safe",
    "sanitation_safe"
]

eda_only_features = [
    "life_expectancy",
    "neonatal_mortality",
    "population_total",
    "urban_population_total"
]

excluded_from_modeling = [
    "gini",
    "adult_literacy"
]

print("Core main features:", core_main_features)
print("Extended features:", extended_features)
print("EDA only features:", eda_only_features)
print("Excluded from modeling:", excluded_from_modeling)

## GDP Variable Choice

For the main model, we retain **GDP per capita (constant 2015 US$)** as the preferred income measure because it is inflation-adjusted and easier to interpret over time.

**GDP per capita PPP (current international $)** is retained only as an extended or robustness feature, not as a core main-model predictor.

---

# End of Stage 2 — Data Cleaning, Data Quality Audit, and Early EDA

Stage 2 examined the quality and usability of the prepared country-year candidate panel.

## Completed in Stage 2
- checked duplicate country-year rows
- assessed missingness across variables
- evaluated target availability over time
- evaluated predictor coverage over time
- assessed joint completeness of the target and core predictors by year
- justified the final modeling window using data evidence
- reviewed feature quality and proposed feature groups
- examined correlations among the target and core predictors
- checked the distribution of under-5 mortality on the raw and log scales

## Main conclusions
- the full 1960–2025 horizon is not jointly usable for the main multivariate model
- joint completeness becomes usable from 2000 onward
- 2024 and 2025 are not suitable target years for evaluated forecasting
- the 2000–2023 period is the most defensible final modeling window
- the raw under-5 mortality distribution is strongly right-skewed
- a log transformation of the target is justified

---

# Stage 3 — Final Modeling Sample Construction